# 04 — LangChain Agent Math Solver (Text-to-Math case study)

Companion notebook to `05-agents-and-tools-langchain-agent-case-study.md`, recreating the personal
**Text-to-Math Problem Solver** project conceptually: a calculator tool, a ReAct-style reasoning
loop, and how it plugs into LangChain's agent framework.

This notebook runs fully **offline by default** -- the mock reasoning loop below executes with no
API key. A guarded, optional section at the bottom shows how to run the *real* LangChain agent
against Groq's Gemma2-9b if you set a `GROQ_API_KEY`, but nothing here requires it.

## Step 1: A safe calculator tool

The one piece of this pipeline that *must* be exact is arithmetic -- so we hand it to real code, not
the LLM (see Chapter 1: LLMs are next-token predictors, not calculators). We use Python's `ast`
module to evaluate expressions safely instead of raw `eval`.

In [ ]:
import ast
import operator as op

_ALLOWED_OPERATORS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg, ast.Mod: op.mod,
}

def _eval_node(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")

def calculator(expression: str) -> str:
    """Safely evaluate a basic arithmetic expression, e.g. '0.15 * 340 + 12'.
    This is the tool description an LLM-based agent would see and use to
    decide when this tool is relevant -- see Chapter 5 on tool descriptions
    functioning as prompts in their own right.
    """
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_eval_node(tree.body))
    except Exception as e:
        return f"Error: {e}"

# quick sanity check
print(calculator("0.15 * 340 + 12"))
print(calculator("60 / (45/60)"))

## Step 2: A word-problem parser tool (mocked)

In the real project this step leans on the LLM's language understanding to turn a word problem into
an arithmetic expression. Here we mock it with a tiny lookup so the rest of the notebook runs
offline -- swap this for a real LLM call (via the LCEL chain pattern from notebook 02) when you have
credentials.

In [ ]:
_MOCK_PARSES = {
    "a train travels 60 miles in 45 minutes. what is its speed in mph?": "60 / (45/60)",
    "what is 15% of 340, then add 12?": "0.15 * 340 + 12",
    "a store gives a 20% discount on a $50 item. what is the final price?": "50 - (0.20 * 50)",
}

def word_problem_parser(problem: str) -> str:
    """Restate a math word problem as a plain arithmetic expression, without solving it.
    Mocked here with a lookup table; in production this would itself be an LLM
    call (a nested chain) -- see Chapter 5.
    """
    key = problem.strip().lower()
    return _MOCK_PARSES.get(key, "UNKNOWN_EXPRESSION")

## Step 3: The ReAct loop, made explicit

Real LangChain agents hide this loop inside `AgentExecutor`. To make the *mechanics* completely
clear before we touch any framework code, here is the same Thought -> Action -> Observation loop
written out by hand, using our two tools above and a scripted "reasoning" function standing in for
the LLM's decisions.

In [ ]:
TOOLS = {
    "word_problem_parser": word_problem_parser,
    "calculator": calculator,
}

def mock_react_agent(problem: str, max_steps: int = 4) -> str:
    """A hand-rolled ReAct loop: at each step, decide an action, run it, observe
    the result, and feed it back in -- exactly the loop AgentExecutor automates
    in real LangChain (see Step 4 below).
    """
    trace = []
    trace.append(f"Thought: I should first turn this word problem into an arithmetic expression.")
    trace.append(f"Action: word_problem_parser[{problem!r}]")
    expression = TOOLS["word_problem_parser"](problem)
    trace.append(f"Observation: {expression}")

    if expression == "UNKNOWN_EXPRESSION":
        trace.append("Final Answer: I don't know how to parse this problem (mock parser has no entry for it).")
        return "\n".join(trace)

    trace.append(f"Thought: Now I should evaluate the expression '{expression}' with the calculator.")
    trace.append(f"Action: calculator[{expression!r}]")
    result = TOOLS["calculator"](expression)
    trace.append(f"Observation: {result}")

    trace.append(f"Thought: I have the final numeric answer.")
    trace.append(f"Final Answer: {result}")
    return "\n".join(trace)

print(mock_react_agent("A train travels 60 miles in 45 minutes. What is its speed in mph?"))

In [ ]:
print(mock_react_agent("What is 15% of 340, then add 12?"))

## Step 4: How this plugs into LangChain's real agent framework

The hand-rolled loop above is exactly what `AgentExecutor` automates when you use a real LLM. If
`langchain` and `langchain-core` are installed, this cell wires the same two tools into a real
LangChain agent object -- but note it only **runs** the executor in the guarded block further down,
since actually invoking it requires a real chat model.

In [ ]:
try:
    from langchain_core.tools import tool

    @tool
    def calculator_tool(expression: str) -> str:
        """Evaluate a basic arithmetic expression, e.g. '0.15 * 340 + 12'.
        Use this whenever the problem requires a numeric calculation."""
        return calculator(expression)

    @tool
    def word_problem_parser_tool(problem: str) -> str:
        """Restate a math word problem as a plain arithmetic expression, without solving it."""
        return word_problem_parser(problem)

    LC_TOOLS = [word_problem_parser_tool, calculator_tool]
    print("LangChain tool objects created:", [t.name for t in LC_TOOLS])
except ImportError:
    LC_TOOLS = None
    print("langchain-core not installed -- install with `pip install langchain langchain-core` "
          "to build real Tool objects. The mock_react_agent() loop above already demonstrates "
          "the mechanics without this dependency.")

## Step 5 (optional, guarded): run the real agent against Groq's Gemma2-9b

This block does **nothing** unless you (a) have `langchain-groq` installed and (b) have set a
`GROQ_API_KEY` environment variable -- exactly the setup used by the actual Text-to-Math project.
It is safe to run this cell with no credentials: it will simply print a message and fall back to
the mock loop from Step 3, so the notebook always completes end-to-end offline.

In [ ]:
import os

def run_real_agent_if_configured(problem: str) -> str:
    """Runs the real LangChain + Groq/Gemma2-9b ReAct agent if credentials are
    available; otherwise falls back to the offline mock agent so this notebook
    never fails without an API key.
    """
    if not os.getenv("GROQ_API_KEY"):
        print("GROQ_API_KEY not set -- falling back to the offline mock agent.\n")
        return mock_react_agent(problem)

    try:
        from langchain_groq import ChatGroq
        from langchain.agents import create_react_agent, AgentExecutor
        from langchain_core.prompts import PromptTemplate
    except ImportError:
        print("GROQ_API_KEY is set, but langchain-groq / langchain isn't installed. "
              "Install with `pip install langchain langchain-groq`. Falling back to mock agent.\n")
        return mock_react_agent(problem)

    llm = ChatGroq(model="gemma2-9b-it", temperature=0)

    react_prompt = PromptTemplate.from_template(
        "Answer the following math word problem as best you can. "
        "You have access to these tools:\n{tools}\n\n"
        "Use this format:\nThought: ...\nAction: {tool_names}\n"
        "Action Input: ...\nObservation: ...\n... (repeat)\nFinal Answer: ...\n\n"
        "Question: {input}\n{agent_scratchpad}"
    )

    agent = create_react_agent(llm, LC_TOOLS, react_prompt)
    executor = AgentExecutor(agent=agent, tools=LC_TOOLS, verbose=True, max_iterations=6)
    result = executor.invoke({"input": problem})
    return result["output"]


if __name__ == "__main__" and os.getenv("GROQ_API_KEY"):
    print(run_real_agent_if_configured("A train travels 60 miles in 45 minutes. What is its speed in mph?"))
else:
    # always executes something, offline-safe
    print(run_real_agent_if_configured("A train travels 60 miles in 45 minutes. What is its speed in mph?"))

## Recap

- `calculator()` is the deterministic tool doing the one thing that must be numerically exact.
- `word_problem_parser()` stands in for the LLM's language-understanding half of the job.
- `mock_react_agent()` makes the Thought -> Action -> Observation loop explicit and runnable offline.
- `run_real_agent_if_configured()` shows exactly how the same tools plug into a real LangChain
  `AgentExecutor` powered by Groq's Gemma2-9b, guarded so it never breaks this notebook when no API
  key is present.

See `05-agents-and-tools-langchain-agent-case-study.md` for the full write-up, including why Groq's
inference speed specifically matters for multi-step agent loops like this one.